# 6. Tool Usage

Notes for this chapter:

- 

In [5]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5-5"

In [6]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stream=False):
    params = {
        "model": model,
        "max_tokens": 16000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if stream:
        with client.messages.stream(**params) as response:
            for text in response.text_stream:
                print(text, end="", flush=True)
            message = response.get_final_message()
    else:
        message = client.messages.create(**params)

    # Responses can start with a thinking block, so find the text block
    for block in message.content:
        if block.type == "text":
            return block.text
    raise RuntimeError(f"No text in response (stop_reason={message.stop_reason!r})")

In [7]:
messages = []

add_user_message(messages, "Hello!")
answer = chat(messages)
add_assistant_message(messages, answer)

answer

'Hello! How can I help you today?'

## Exercises

## Tool Functions

Plain python functions should be well named, and their arguments should be clear. Well worded errors help too

In [8]:


from datetime import datetime


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

print(get_current_datetime())
print(get_current_datetime("%H:%M"))


2026-10-22 16:01:50
16:01


## Tool Schemas 

The next step is creating a JSON schema that tells Claude what arguments your function expects and how to use it. This schema acts as documentation that Claude reads to understand when and how to call your tools.

This is not specific to Claude.


Best practices include:

Aim for 3-4 sentences explaining what the tool does
Describe when Claude should use it
Explain what kind of data it returns
Provide detailed descriptions for each argument


## A functiona schema hack:

Instead of writing JSON schemas from scratch, you can use Claude itself to generate them. Here's the process:

- Copy your tool function code
- Go to Claude and ask it to write a JSON schema for tool calling
- Include the Anthropic documentation on tool use as context
- Let Claude generate a properly formatted schema following best practices

In [9]:
get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}

Adding type safety

In [10]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

## Using the tool

In [20]:
from anthropic.types import TextBlock

tools = { "get_current_datetime": get_current_datetime }

messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

add_assistant_message(messages, response.content)

if response.stop_reason == "tool_use":
    tool_results = []
    for block in response.content:
        if block.type != "tool_use":
            continue
        try:
            date_format = block.input.get("date_format", "%Y-%m-%d %H:%M:%S")
            if not isinstance(date_format, str):
                raise ValueError(f"date_format must be a string, got {type(date_format).__name__}")
            output = tools[block.name](date_format)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(output)
            })
        except Exception as e:
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": f"Error: {e}",
                "is_error": True
            })

    add_user_message(messages, tool_results)

response = client.messages.create(
    model=model,
    max_tokens=16000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

for block in response.content:
    if block.type == "text":
        print(block.text)




The current time is **11:07:27**.


## Multi-turn conversations

An insight into how we build multi-turn conversations, where multiple tool calls might be used. For example:


User asks: "What day is 103 days from today?"
Claude responds with a tool use block requesting get_current_datetime
Your server calls the function and returns the result
Claude realizes it needs more information and requests add_duration_to_datetime
Your server calls that function and returns the result
Claude now has enough information to provide the final answer

To do so, we need to handle the looping and call each tool as required:

```python
def run_conversation(messages):
    while True:
        response = chat(messages)

        add_assistant_message(messages, response)

        # Pseudo code
        if response isn't asking for a tool:
            break

        tool_result_blocks = run_tools(response)
        add_user_message(messages, tool_result_blocks)
        
    return messages
```

In [33]:
import json
from anthropic.types import Message, ToolParam
import re
from datetime import datetime as DateTime, timedelta

DURATION_UNITS = {"w": "weeks", "d": "days", "h": "hours", "m": "minutes", "s": "seconds"}

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)



def chat(messages, system=None, stream=False, tools=None):
    params = {
        "model": model,
        "max_tokens": 16000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools


    return client.messages.create(**params)

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )


def add_duration_to_datetime(datetime, duration):
    start = DateTime.fromisoformat(datetime)

    # Split a duration like "1w 2d 3h 30m" into (number, unit) pairs
    compact = duration.replace(" ", "").lower()
    parts = re.findall(r"(-?\d+)([wdhms])", compact)
    if not parts or "".join(number + unit for number, unit in parts) != compact:
        raise ValueError(f"Invalid duration {duration!r}. Use a number and unit, e.g. '2h30m' or '1w 3d'.")

    delta = timedelta(**{DURATION_UNITS[unit]: int(number) for number, unit in parts})
    return (start + delta).isoformat()


reminders = []

def set_reminder(reminder, datetime):
    if not reminder.strip():
        raise ValueError("reminder cannot be empty")

    when = DateTime.fromisoformat(datetime)
    reminders.append({"reminder": reminder, "datetime": when.isoformat()})
    return f"Reminder set for {when:%A %d %B %Y at %H:%M}: {reminder}"
    

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)

add_duration_to_datetime_schema = ToolParam({
      "name": "add_duration_to_datetime",
      "description": "Adds a duration to a date and time and returns the resulting date and time. Use this to work out dates like 'in 3 days' or'two hours from now'. Call get_current_datetime first if the starting point is now.",
      "input_schema": {
          "type": "object",
          "properties": {
              "datetime": {
                  "type": "string",
                  "description": "The starting date and time in ISO 8601 format, e.g. '2026-10-23T14:30:00' or '2026-10-23'."
              },
              "duration": {
                  "type": "string",
                  "description": "The amount of time to add, as numbers with units: w (weeks), d (days), h (hours), m (minutes), s (seconds). Combine units, e.g. '2h30m' or '1w 3d'. Use a negative number to subtract, e.g. '-15m'."
              }
          },
          "required": ["datetime", "duration"]
      }
  })

set_reminder_schema = ToolParam({
      "name": "set_reminder",
      "description": "Sets a reminder for the user at a specific date and time. Returns a confirmation message.",
      "input_schema": {
          "type": "object",
          "properties": {
              "reminder": {
                  "type": "string",
                  "description": "What to remind the user about, e.g. 'Dentist appointment'."
              },
              "datetime": {
                  "type": "string",
                  "description": "When to send the reminder, in ISO 8601 format, e.g. '2026-10-30T09:15:00'."
              }
          },
          "required": ["reminder", "datetime"]
      }
  })


def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tool_result_blocks = []
    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result", 
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

tools_schemas = [
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema
]

def run_conversation(messages):
    while True:
        response = chat(messages, tools=tools_schemas)
        add_assistant_message(messages, response)
        print(text_from_message(response))
        
        if response.stop_reason != "tool_use":
            break
            
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    
    return messages

tools = {
      "get_current_datetime": get_current_datetime,
      "add_duration_to_datetime": add_duration_to_datetime,
      "set_reminder": set_reminder,
}  



messages = []

add_user_message(messages, "Remind me that I have the dentist in 2 days time")

run_conversation(messages)

for message in messages:
    print(message)


print(reminders)




I've set your reminder for "Dentist appointment" on Sunday, 25 October 2026 at 11:31, which is two days from now.

If you want the reminder earlier that day, or the night before, I can change it.
{'role': 'user', 'content': 'Remind me that I have the dentist in 2 days time'}
{'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01HYGdnvFgPYygvDiWLbf6Fg', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use', toolset_name=None)]}
{'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01HYGdnvFgPYygvDiWLbf6Fg', 'content': '"2026-10-23 11:31:16"', 'is_error': False}]}
{'role': 'assistant', 'content': [ToolUseBlock(id='toolu_018VBRCgANFsSPr9ruimZxxS', caller=DirectCaller(type='direct'), input={'datetime': '2026-10-23T11:31:16', 'duration': '2d'}, name='add_duration_to_datetime', type='tool_use', toolset_name=None)]}
{'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_018VBRCgANFsSPr9ruimZxxS', 'content': 

Tool usage can also be streamed, with or without json validation. Use `fine_grained=True` to enable without json validation. When enabled, we'll need to handle parsing incorrect json on our end. With json validation, we'll get valid json but will have to wait for key,value pairs to be completed. 

## Built in tools

### Text Edit 

The schema comes baked in, but we need to provide the function implementation

### Web Search

```
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search", 
    "max_uses": 5
}
```

Must be enabled in capabilties, but the API can web search and returns multiple blocks:

When Claude uses the web search tool, the response contains several types of blocks:

- Text blocks - Claude's explanation of what it's doing
- ServerToolUseBlock - Shows the exact search query Claude used
- WebSearchToolResultBlock - Contains the search results
- WebSearchResultBlock - Individual search results with titles and URLs
- Citation blocks - Text that supports Claude's statements

`allowed_domains` can be added to the schema to restrict the sources.

